## **Part One**

### The data contains stock returns of small retailers

### An analyst claims there is **a bias in moment conditions of instrumental variables** such that **$Z^T$(Y - XB) =  $\sigma \begin{bmatrix} 1 \\ 1 \\ 1 \end{bmatrix}$**


### **1.** Update the GMM model by incorporating the $\sigma$ term to the instrumental variable moment expressions.

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.sandbox.regression.gmm import GMM

In [2]:
input_table = pd.read_csv("/Users/lucasben/Documents/mba-business-analytics/courses/predictive-modelling/midterm-project/midterm_partone.csv")

In [3]:
model_iv = sm.OLS(input_table["Inventory Turnover"],input_table[["Constant","Current Ratio","Quick Ratio",\
                                                                 "Debt Asset Ratio"]]).fit()
endog_predict = model_iv.predict(input_table[["Constant","Current Ratio","Quick Ratio","Debt Asset Ratio"]])
input_table["Endogenous Param"] = endog_predict

In [4]:
model_2sls = sm.OLS(input_table["Stock Change"], input_table[["Constant","Endogenous Param",\
                                                              "Operating Profit","Interaction Effect",\
                                                             ]]).fit()
model_2sls.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:           Stock Change   R-squared:                       0.015
Model:                            OLS   Adj. R-squared:                  0.013
Method:                 Least Squares   F-statistic:                     8.530
Date:                Sat, 08 Nov 2025   Prob (F-statistic):           1.27e-05
Time:                        14:28:37   Log-Likelihood:                -1186.5
No. Observations:                1696   AIC:                             2381.
Df Residuals:                    1692   BIC:                             2403.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
======================================================================================
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Constant              -0.0176      0.020     -0.896      0.370      -0.056       0.021
Endogenous Param       0.0011      0.001      1.827      0.068   -7.76e-05       0.002
Operating Profit      -0.1201      0.028     -4.319      0.000      -0.175      -0.066
Interaction Effect     0.0014      0.000      3.621      0.000       0.001       0.002
==============================================================================
Omnibus:                      368.832   Durbin-Watson:                   2.243
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             3433.920
Skew:                           0.742   Prob(JB):                         0.00
Kurtosis:                       9.811   Cond. No.                         109.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [5]:
y  = np.array(input_table["Stock Change"])
X = np.array(input_table[["Inventory Turnover","Operating Profit","Interaction Effect"]])
Z = np.array(input_table[["Current Ratio","Quick Ratio","Debt Asset Ratio"]])


In [7]:
class gmm(GMM):
    def __init__(self, *args, delta=0.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.delta = delta 

    def momcond(self, params):
        p0, p1, p2, p3 = params
        endog = self.endog
        exog = self.exog
        inst = self.instrument   

        error0 = endog - p0 - p1 * exog[:,0] - p2 * exog[:,1] - p3 * exog[:,2]
        error1 = (endog - p0 - p1 * exog[:,0] - p2 * exog[:,1] - p3 * exog[:,2]) * exog[:,1]
        error2 = (endog - p0 - p1 * exog[:,0] - p2 * exog[:,1] - p3 * exog[:,2]) * exog[:,2]
        error3 = (endog - p0 - p1 * exog[:,0] - p2 * exog[:,1] - p3 * exog[:,2]) * inst[:,0] - self.delta
        error4 = (endog - p0 - p1 * exog[:,0] - p2 * exog[:,1] - p3 * exog[:,2]) * inst[:,1] - self.delta
        error5 = (endog - p0 - p1 * exog[:,0] - p2 * exog[:,1] - p3 * exog[:,2]) * inst[:,2] - self.delta

        g = np.column_stack((error0, error1, error2, error3, error4, error5))
        return g


beta0 = np.array([0.1, 0.1, 0.1, 0.1])
res = gmm(endog = y, exog = X, instrument = Z, k_moms=6, k_params=4, delta=0.5).fit(beta0)

res.summary()

Optimization terminated successfully.
         Current function value: 0.324795
         Iterations: 9
         Function evaluations: 13
         Gradient evaluations: 13
Optimization terminated successfully.
         Current function value: 1.042706
         Iterations: 10
         Function evaluations: 13
         Gradient evaluations: 13
Optimization terminated successfully.
         Current function value: 0.714303
         Iterations: 7
         Function evaluations: 10
         Gradient evaluations: 10
Optimization terminated successfully.
         Current function value: 0.676510
         Iterations: 6
         Function evaluations: 10
         Gradient evaluations: 10
Optimization terminated successfully.
         Current function value: 0.958586
         Iterations: 8
         Function evaluations: 12
         Gradient evaluations: 12
Optimization terminated successfully.
         Current function value: 5.552729
         Iterations: 12
         Function evaluations: 17
      

<class 'statsmodels.iolib.summary.Summary'>
"""
                                 gmm Results                                  
==============================================================================
Dep. Variable:                      y   Hansen J:                        272.2
Model:                            gmm   Prob (Hansen J):              7.75e-60
Method:                           GMM                                         
Date:                Sat, 08 Nov 2025                                         
Time:                        14:34:12                                         
No. Observations:                1696                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
p 0           -2.6875      0.544     -4.940      0.000      -3.754      -1.621
p 1            0.0095      0.009      1.109      0.268      -0.007       0.026
p 2           11.9082      2.136      5.575      0.000       7.722      16.094
p 3           -0.0862      0.021     -4.011      0.000      -0.128      -0.044
==============================================================================
"""

### **2.** Analyze the GMM summary table & test statistics of coefficients & determine if the analyst's claim is statistically justified.